In [14]:
from selenium import webdriver 
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time,re
import pandas as pd

service = Service(ChromeDriverManager().install())
options = Options()
# options.add_argument("--headless")
# options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--start-maximized")
options.add_argument("--window-size=1920x1080")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")

driver = webdriver.Chrome(service=service, options=options)

category_urls = {
    "여성의류":"https://display.wconcept.co.kr/rn/best?displayCategoryType=10101&displaySubCategoryType=ALL&gnbType=Y",
    "가방":"https://display.wconcept.co.kr/rn/best?displayCategoryType=10102&displaySubCategoryType=ALL&gnbType=Y",
    "신발":"https://display.wconcept.co.kr/rn/best?displayCategoryType=10103&displaySubCategoryType=ALL&gnbType=Y",
    "엑세서리":"https://display.wconcept.co.kr/rn/best?displayCategoryType=10104&displaySubCategoryType=ALL&gnbType=Y"
}
all_data = []

def extract_product_id_from_imgsrc(src: str) -> str | None:
    if not src: 
        return None 
    m = re.search(r"/(\d+)(?:[_\.][^/?]+)?(?:\?.*)?$", src)
    return m.group(1) if m else None 

def build_product_url(pid: str| None) -> str : 
    return  f"https://www.wconcept.co.kr/Product/{pid}" if pid else "" 

for cat_name, url in category_urls.items() : 
    driver.get(url)
    time.sleep(2)

    for _ in range(2) : 
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1)
        
    products = driver.find_elements(By.CSS_SELECTOR, "div.product-item.item.type-all")[:100]
    for product in products : 
        try: 
            name = product.find_element(By.CSS_SELECTOR, "span.prdc-title span.text.detail").text.strip()
            try:
                img = product.find_element(By.CSS_SELECTOR,"button.area-img img")
                src = img.get_attribute('src')
                pid = extract_product_id_from_imgsrc(src)
                url = build_product_url(pid)
            except :
                url = ""
            try:
                review_info = product.find_element(By.CSS_SELECTOR,"span.area-info span.text.cnt").text
                review_count = int(re.sub(r"[^\d]","",review_info)) if review_info else 0 
            except:
                review_count = 0 

            try :
                rating_text = product.find_element(By.CSS_SELECTOR,"span.area-info em.score").text
                review_avg = float(re.findall(r"[0-9.]+",rating_text)[0])
            except:
                review_avg = 0.0

            all_data.append({
                "카테고리": cat_name,
                "상품명": name,
                "URL": url,
                "리뷰 수": review_count,
                "평균평점": review_avg
            })
                
        except Exception as e :
            print("에러발생 :",e)
            continue 

driver.quit()

df = pd.DataFrame(all_data)
df_top100 = df.sort_values(by="리뷰 수",ascending=False).head(100).reset_index(drop=True)

category_summary = df.groupby("카테고리").agg({
    "리뷰 수"    :"sum", 
    "평균평점"   :"mean"
}).reset_index()

print(f""" 
상위 100개 상품:
{df_top100.head()}

카테고리별 통계:
{category_summary}
""")


 
상위 100개 상품:
   카테고리                                    상품명  \
0    신발                OORIGINAL BLACK - 조리 블랙   
1    가방  [14%쿠폰] [단독] Panier Petit Bag_6colors   
2  엑세서리   [단독] Round Logo Ball Cap (10 Colors)   
3  엑세서리           [단독]BT8032 티디 스퀘어 버클 벨트 18mm   
4  여성의류                       실켓 코튼 티셔츠_8color   

                                            URL  리뷰 수  평균평점  
0  https://www.wconcept.co.kr/Product/301721294  8478   5.0  
1  https://www.wconcept.co.kr/Product/301756891  3730   5.0  
2  https://www.wconcept.co.kr/Product/301628486  3682   4.9  
3  https://www.wconcept.co.kr/Product/304460283  3131   4.9  
4  https://www.wconcept.co.kr/Product/301218440  2968   4.9  

카테고리별 통계:
   카테고리   리뷰 수   평균평점
0    가방  24218  4.267
1    신발  27342  4.281
2  엑세서리  30519  4.016
3  여성의류  31521  4.435



In [31]:
from selenium import webdriver 
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time,re
import pandas as pd

service = Service(ChromeDriverManager().install())
options = Options()
# options.add_argument("--headless")
# options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--start-maximized")
options.add_argument("--window-size=1920x1080")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")

driver = webdriver.Chrome(service=service, options=options)

for item in all_data:
    url = item["URL"]
    driver.get(f"{url}#review")
    time.sleep(1)
    reviews =driver.find_elements(By.CSS_SELECTOR,"p.pdt_review_text")
    for review in reviews :
        product_review = review.text 

driver.quit()

InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=139.0.7258.128); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	GetHandleVerifier [0x0x6cffc3+65331]
	GetHandleVerifier [0x0x6d0004+65396]
	(No symbol) [0x0x4c3f63]
	(No symbol) [0x0x4b38e0]
	(No symbol) [0x0x4d1709]
	(No symbol) [0x0x537e4c]
	(No symbol) [0x0x5524d9]
	(No symbol) [0x0x5312d6]
	(No symbol) [0x0x500910]
	(No symbol) [0x0x501784]
	GetHandleVerifier [0x0x9138b3+2439203]
	GetHandleVerifier [0x0x90eae2+2419282]
	GetHandleVerifier [0x0x6f712a+225434]
	GetHandleVerifier [0x0x6e6e08+159096]
	GetHandleVerifier [0x0x6edd5d+187597]
	GetHandleVerifier [0x0x6d7ad8+96840]
	GetHandleVerifier [0x0x6d7c62+97234]
	GetHandleVerifier [0x0x6c277a+9962]
	BaseThreadInitThunk [0x0x75f4fcc9+25]
	RtlGetAppC

In [32]:
from selenium import webdriver 
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time

service = Service(ChromeDriverManager().install())
options = Options()
# options.add_argument("--headless")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--start-maximized")
options.add_argument("--window-size=1920x1080")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")

driver = webdriver.Chrome(service=service, options=options)

for item in all_data:
    url = item["URL"]
    driver.get(f"{url}#review")
    time.sleep(1)
    
    reviews = driver.find_elements(By.CSS_SELECTOR, "p.pdt_review_text")
    product_reviews = []
    for review in reviews:
        txt = review.text.strip()
        if txt:
            product_reviews.append(txt)
    
    # 상품명과 함께 출력
    print("="*50)
    print(f"상품명: {item['상품명']}")
    print(f"URL: {url}")
    print("리뷰:")
    for r in product_reviews:
        print("-", r)

driver.quit()


상품명: [72H팝업] [단독]캐시미어 피크드 칼라 더블 롱코트_2colors
URL: https://www.wconcept.co.kr/Product/306038291
리뷰:
- 지난 겨울에 살말 고민만하다 품절되서 구매 못했었는데 이번에 다시 오픈된거 보자마자 빨리 구매했어요!
배송받자마자 입어봤는데 핏도 멋스럽고 소재도 좋네여 ... 역시 고민한 시간이 아깝지 않은 퀄리티에요ㅜㅜ겨울이 기다려져요 ㅎㅎ
- 작년에 멜란지그레이 컬러 구매하고 너무 잘 입어서 프리오픈 떠서 할인 받아 구매했는데 컬러도 고급스럽고 퀄리티 좋아서 오래 입을 수 있을 것 같아요!!
회사로 배송지 선택을 잘못해서.. 회사에서 한 번 입어봤는데 팀원들 다 예쁘다고 해서 좋은 소비한 것 같아 기분 좋았네용..ㅎㅎㅎ
- 작년부터 사고 싶었는데 품절이라 아쉬웠던 제품이라 이번에 미리 알림 걸어두고 구매했어요
클래식하고 오래 입을 수 있는 디자인이라 군더더기 없이 핏이 딱 떨어집니다
소재도 부드러워서 올겨울 내내 잘 입을 것 같아요
- 살짝 크긴하지만 오버핏으로 여유롭게 이뻐요
기장도 긴편인데 뒤에 절개가 들어있어 부담스럽지 않고
소재도 탄탄하고 특히 색감이 너무 맘에들어요!!
- 이거 입으면 사람이 길어보이고 태가 나요!! 소재도
좋고 카라가 커서 얼굴 작아보이는 효과는 덤 ㅎㅎ
올겨울 내내 교복처럼 입고 댕겼어용
- 가격대비 코트 재질이 좋은것 같아요~
약간 오버핏이고 롱이라 저에게는 좀 크고 긴 느낌이지만 재질이 좋고 약간 밝은 회색이라 화사하게 입을수 있을것 같아요~♡
상품명: 샤론 슬림 니트 가디건(멜란지그레이)
URL: https://www.wconcept.co.kr/Product/307350085
리뷰:
- 진짜 얇네요
그래서 여름에 입기 더 좋은거 같아용~~~
소재도 부들부들하고
맘에 들어요
- 소재아주좋습니다 깔별로 구입
길이도좋고 소재가 얇아 오래못입을까봐 걱정 ㅋㅋ
- 소재아주좋습니다 깔별로 구입
길이도좋고 소재가 얇아 오래못입을까봐 걱정 ㅋㅋ
- 소매가 길어서 너무 좋네요 

KeyboardInterrupt: 

In [15]:
from openpyxl import Workbook

wb = Workbook()

ws_top100 = wb.active
ws_top100.title = "상위100상품"

ws_top100.append(["카테고리","상품명","URL","리뷰 수","평균평점"])
for idx,row in df_top100.iterrows():
    ws_top100.append([row["카테고리"],row["상품명"],row["URL"],row["리뷰 수"],row["평균평점"]])

ws_category=wb.create_sheet(title = "카테고리별 통계") 

ws_category.append(["카테고리","총 리뷰수","평균평점"])
for idx, row in category_summary.iterrows():
    ws_category.append([
        row["카테고리"],row["리뷰 수"],round(row["평균평점"],2)
    ])

filename = "wconcept_result.xlsx"
wb.save(filename)
print(f"1개의 엑셀파일에 시트2개 저장 완료 : {filename}")
    

1개의 엑셀파일에 시트2개 저장 완료 : wconcept_result.xlsx


In [23]:
import pandas as pd 
import pymysql
from sqlalchemy import create_engine, text

DB_NAME = "Wconcept_db_v1"
TABLE_TOP100 = "top100_products"
TABLE_CATEGORY = "category_summary"
USER = "root"
PASSWORD = "hithere4961"
HOST = "localhost"
PORT = 3306

xlsx_path = "wconcept_result.xlsx"
SHEET_TOP100 = "상위100상품"
SHEET_CATEGORY = "카테고리별 통계"

df_top100 = pd.read_excel(xlsx_path, sheet_name = SHEET_TOP100)
df_category = pd.read_excel(xlsx_path, sheet_name = SHEET_CATEGORY)

top100_cols = {
    "카테고리": "category",
    "상품명": "product_name",
    "URL": "url",
    "리뷰 수": "review_count",
    "평균평점": "avg_rating",
}

category_cols = {
    "카테고리": "category",
    "총 리뷰수": "total_review_count",
    "평균평점": "avg_rating",
}

df_top100 = df_top100.rename(columns=top100_cols)[list(top100_cols.values())]
df_category = df_category.rename(columns=category_cols)[list(category_cols.values())]

df_top100 = df_top100.dropna(subset=["category","product_name","review_count","avg_rating"])
df_category = df_category.dropna(subset=["category","total_review_count","avg_rating"])

df_top100["review_count"] = df_top100["review_count"].astype(int)
df_top100["avg_rating"] = df_top100["avg_rating"].astype(float).round(2)

df_category["total_review_count"] = df_category["total_review_count"].astype(int)
df_category["avg_rating"] = df_category["avg_rating"].astype(float).round(2)


conn_admin = pymysql.connect(host=HOST,user=USER,password=PASSWORD,port=PORT, autocommit=True)

with conn_admin.cursor() as cur :
    cur.execute(f"CREATE DATABASE IF NOT EXISTS `{DB_NAME}` CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;")
conn_admin.close()

engine = create_engine(
    f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}?charset=utf8mb4"
)

ddl_top100 = f"""
CREATE TABLE IF NOT EXISTS `{TABLE_TOP100}` (
    `id` BIGINT UNSIGNED NOT NULL PRIMARY KEY AUTO_INCREMENT,
    `category` VARCHAR(50) NOT NULL,
    `product_name` VARCHAR(255) NOT NULL,
    `url` TEXT NULL , 
    `review_count` INT NOT NULL,
    `avg_rating` DECIMAL(3,2) NOT NULL,
    `created_at` TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
)
"""


ddl_category = f"""
CREATE TABLE IF NOT EXISTS `{TABLE_CATEGORY}` (
    `id` INT UNSIGNED NOT NULL PRIMARY KEY AUTO_INCREMENT,
    `category` VARCHAR(50) NOT NULL,
    `total_review_count` INT NOT NULL,
    `avg_rating` DECIMAL(3,2) NOT NULL,
    `created_at` TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
)
"""

with engine.begin() as conn : 
    conn.execute(text(ddl_top100))
    conn.execute(text(ddl_category))

with engine.begin() as conn : 
    df_top100[["category","product_name","url","review_count","avg_rating"]].to_sql(
        TABLE_TOP100, con=conn,if_exists="append",index=False
    )
    df_category[["category","total_review_count","avg_rating"]].to_sql(
        TABLE_CATEGORY, con=conn,if_exists="append",index=False
    )

def ensure_index(conn,schema,table,index_name,column_list) :
    q = text(""" 
        SELECT 1 
        FROM information_schema.statistics
        WHERE table_schema = :schema AND 
        table_name=:table AND
        index_name=:index_name
    """)
    exists = conn.execute(q,{"schema":schema,"table":table,"index_name":index_name}).fetchone()
    

with engine.begin() as conn :
    ensure_index(conn, DB_NAME, TABLE_TOP100,f"idx_{TABLE_TOP100}_category",["category"])
    ensure_index(conn, DB_NAME, TABLE_TOP100,f"idx_{TABLE_TOP100}_review_count",["review_count"])
    ensure_index(conn, DB_NAME, TABLE_TOP100,f"idx_{TABLE_TOP100}_avg_rating",["avg_rating"])
    ensure_index(conn, DB_NAME, TABLE_CATEGORY,f"idx_{TABLE_CATEGORY}_total_review",["total_review_count"])
    ensure_index(conn, DB_NAME, TABLE_CATEGORY,f"idx_{TABLE_CATEGORY}_avg_rating",["avg_rating"])

print("MySQL 데이터 생성 완료")


MySQL 데이터 생성 완료


In [33]:
import hashlib
from sqlalchemy import text

PRODUCT_TABLE = "top100_products"
REVIEWS_TABLE = "reviews"

def _md5(product_id: int, s: str) -> str:
    return hashlib.md5(f"{product_id}|{s}".encode("utf-8")).hexdigest()

def _normalize(txt: str) -> str:
    # 공백/개행 정리(선택)
    return " ".join(txt.split()).strip()

# URL -> product_id 매핑 (이미 products를 채워두셨으므로 URL로 찾는게 가장 확실)
with engine.begin() as conn:
    # URL이 TEXT여도 where 조건으로 조회는 문제없음
    sql_get_id = text(f"SELECT id, category, product_name FROM {PRODUCT_TABLE} WHERE url = :url LIMIT 1")

    ins_sql = text(f"""
        INSERT IGNORE INTO {REVIEWS_TABLE}
            (product_id, review_text, review_md5, product_name, category)
        VALUES
            (:product_id, :review_text, :review_md5, :product_name, :category)
    """)

    for item in all_data:
        url = item.get("URL")
        if not url:
            continue

        row = conn.execute(sql_get_id, {"url": url}).fetchone()
        if not row:
            # 필요한 경우: 제품 테이블에 없다면 먼저 upsert 하도록 변경 가능
            print(f"[SKIP] products에 없음: {url}")
            continue

        pid, category, pname = int(row[0]), row[1], row[2]

        for raw in item.get("리뷰", []):
            txt = _normalize(raw)
            if not txt:
                continue
            conn.execute(ins_sql, {
                "product_id": pid,
                "review_text": txt,
                "review_md5": _md5(pid, txt),
                "product_name": pname,   # 옵션(편의 저장)
                "category": category     # 옵션(편의 저장)
            })

print("✅ 리뷰 삽입 완료")


[SKIP] products에 없음: https://www.wconcept.co.kr/Product/306038291
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/307350085
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/306038292
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/307335695
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/307350064
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/306141668
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/307363913
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/306788003
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/301864566
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/307350088
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/304891098
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/307059292
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/301472489
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/306769137
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/306042608
[SKIP] pro

In [34]:
import re, hashlib
from urllib.parse import urlparse, parse_qsl, urlencode, urlunparse
from sqlalchemy import text

PRODUCT_TABLE = "top100_products"
REVIEWS_TABLE  = "reviews"

def normalize_url(u: str) -> str:
    if not u: return u
    u = u.strip()
    # 1) 해시(#...) 제거
    u = u.split('#', 1)[0]
    # 2) 공백/개행 제거
    u = " ".join(u.split())
    # 3) 쿼리 파라미터 정렬(순서 달라져도 동일 취급)
    try:
        p = urlparse(u)
        q = sorted(parse_qsl(p.query, keep_blank_values=True))
        new_query = urlencode(q)
        # 4) 도메인 소문자
        netloc = p.netloc.lower()
        # 5) 끝 슬래시 정규화(원하면 제거)
        path = re.sub(r'//+', '/', p.path or '')
        # 재조합
        u = urlunparse((p.scheme, netloc, path, '', new_query, ''))
        # 끝 슬래시 제거(선택): 사이트 규칙에 맞게 결정
        if u.endswith('/'):
            u = u[:-1]
    except Exception:
        pass
    return u

with engine.begin() as conn:
    sql_exact = text(f"""
        SELECT id, category, product_name, url
        FROM {PRODUCT_TABLE}
        WHERE url = :url
        LIMIT 1
    """)
    sql_exact_norm = text(f"""
        SELECT id, category, product_name, url
        FROM {PRODUCT_TABLE}
        WHERE REPLACE(SUBSTRING_INDEX(url, '#', 1), ' ', '') = :norm     -- 해시/공백 제거 일치
        LIMIT 1
    """)
    sql_like = text(f"""
        SELECT id, category, product_name, url
        FROM {PRODUCT_TABLE}
        WHERE url LIKE :prefix
        ORDER BY LENGTH(url) DESC
        LIMIT 1
    """)
    sql_by_name = text(f"""
        SELECT id, category, product_name, url
        FROM {PRODUCT_TABLE}
        WHERE product_name = :name
        LIMIT 1
    """)

    ins_review = text(f"""
        INSERT IGNORE INTO {REVIEWS_TABLE}
            (product_id, review_text, review_md5, product_name, category)
        VALUES
            (:product_id, :review_text, :review_md5, :product_name, :category)
    """)

    miss = 0
    for item in all_data:
        raw = (item.get("URL") or "").strip()
        if not raw:
            continue

        norm = normalize_url(raw)

        row = conn.execute(sql_exact, {"url": raw}).fetchone()
        if not row:
            row = conn.execute(sql_exact_norm, {"norm": normalize_url(raw)}).fetchone()
        if not row:
            # 도메인+경로까지만 LIKE로 폴백
            # 예) https://site.com/p/123?x=1 → https://site.com/p/123%
            parsed = urlparse(norm)
            prefix = urlunparse((parsed.scheme, parsed.netloc, parsed.path, '', '', ''))
            if prefix.endswith('/'): prefix = prefix[:-1]
            row = conn.execute(sql_like, {"prefix": prefix + '%'}).fetchone()
        if not row:
            # 최후 폴백: 상품명 일치
            row = conn.execute(sql_by_name, {"name": item.get("상품명")}).fetchone()

        if not row:
            miss += 1
            print(f"[SKIP] products에 없음: {raw}  → norm={norm}")
            continue

        pid, category, pname, db_url = int(row[0]), row[1], row[2], row[3]

        for t in item.get("리뷰", []):
            t = " ".join(t.split()).strip()
            if not t: 
                continue
            md5key = hashlib.md5(f"{pid}|{t}".encode("utf-8")).hexdigest()
            conn.execute(ins_review, {
                "product_id": pid,
                "review_text": t,
                "review_md5": md5key,
                "product_name": pname,
                "category": category
            })

    print("완료. 매칭 실패 건수:", miss)


[SKIP] products에 없음: https://www.wconcept.co.kr/Product/306038291  → norm=https://www.wconcept.co.kr/Product/306038291
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/307350085  → norm=https://www.wconcept.co.kr/Product/307350085
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/306038292  → norm=https://www.wconcept.co.kr/Product/306038292
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/307335695  → norm=https://www.wconcept.co.kr/Product/307335695
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/307350064  → norm=https://www.wconcept.co.kr/Product/307350064
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/306141668  → norm=https://www.wconcept.co.kr/Product/306141668
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/307363913  → norm=https://www.wconcept.co.kr/Product/307363913
[SKIP] products에 없음: https://www.wconcept.co.kr/Product/306788003  → norm=https://www.wconcept.co.kr/Product/306788003
[SKIP] products에 없음: https://www.wconcept.co.kr/

In [35]:
import re, hashlib
from sqlalchemy import text

def extract_code(u: str):
    m = re.search(r'(\d+)$', u or '')
    return int(m.group(1)) if m else None

sql_by_code = text("""
    SELECT id, category, product_name FROM top100_products
    WHERE product_code = :code
    LIMIT 1
""")

ins_review = text("""
    INSERT IGNORE INTO reviews
      (product_id, review_text, review_md5, product_name, category)
    VALUES
      (:product_id, :review_text, :review_md5, :product_name, :category)
""")

with engine.begin() as conn:
    miss = 0
    for item in all_data:
        code = extract_code(item.get("URL"))
        if not code:
            print("[SKIP] 코드 추출 실패:", item.get("URL"))
            miss += 1
            continue

        row = conn.execute(sql_by_code, {"code": code}).fetchone()
        if not row:
            print("[SKIP] products에 코드 없음:", code)
            miss += 1
            continue

        pid, category, pname = int(row[0]), row[1], row[2]

        for t in item.get("리뷰", []):
            t = " ".join(t.split()).strip()
            if not t: 
                continue
            md5key = hashlib.md5(f"{pid}|{t}".encode("utf-8")).hexdigest()
            conn.execute(ins_review, {
                "product_id": pid,
                "review_text": t,
                "review_md5": md5key,
                "product_name": pname,
                "category": category
            })

    print("완료. 매칭 실패 건수:", miss)


[SKIP] products에 코드 없음: 306038291
[SKIP] products에 코드 없음: 307350085
[SKIP] products에 코드 없음: 306038292
[SKIP] products에 코드 없음: 307335695
[SKIP] products에 코드 없음: 307350064
[SKIP] products에 코드 없음: 306141668
[SKIP] products에 코드 없음: 307363913
[SKIP] products에 코드 없음: 306788003
[SKIP] products에 코드 없음: 301864566
[SKIP] products에 코드 없음: 307350088
[SKIP] products에 코드 없음: 304891098
[SKIP] products에 코드 없음: 307059292
[SKIP] products에 코드 없음: 301472489
[SKIP] products에 코드 없음: 306769137
[SKIP] products에 코드 없음: 306042608
[SKIP] products에 코드 없음: 307381191
[SKIP] products에 코드 없음: 307350112
[SKIP] products에 코드 없음: 307269221
[SKIP] products에 코드 없음: 307377173
[SKIP] products에 코드 없음: 307375882
[SKIP] products에 코드 없음: 306567458
[SKIP] products에 코드 없음: 306879339
[SKIP] products에 코드 없음: 307360523
[SKIP] products에 코드 없음: 302266402
[SKIP] products에 코드 없음: 306873884
[SKIP] products에 코드 없음: 307173338
[SKIP] products에 코드 없음: 306822419
[SKIP] products에 코드 없음: 306937536
[SKIP] products에 코드 없음: 306931367
[SKIP] product